# Full Synthetic Figure Reproduction

This is the main notebook for reproducing the paper's model-generated figures. Fig.5 remains the priority, but Fig.2-Fig.4 are also organized as figure-level workflows. Empirical MEG figures require data that are not distributed in this repository.

In [ ]:
from pathlib import Path
import sys

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / "python"))

import matplotlib.pyplot as plt
import pandas as pd

from criticality_analysis import (
    avalanche_table,
    find_output_file,
    fit_power_law,
    load_medie,
    load_q,
    load_rate,
    load_spikes,
    make_parameter_sweep,
    run_simulation,
    summarize_run,
)
from criticality_analysis.plotting import (
    plot_avalanche_distribution,
    plot_hysteresis,
    plot_metric_heatmap,
    plot_module_rate,
    plot_overlap,
    plot_raster,
)

FIG_DIR = PROJECT / "results" / "figures"
TABLE_DIR = PROJECT / "results" / "tables"
RUN_ROOT = PROJECT / "results" / "runs"
GENERATED_CONFIGS = PROJECT / "results" / "generated_configs"
for path in [FIG_DIR, TABLE_DIR, RUN_ROOT, GENERATED_CONFIGS]:
    path.mkdir(parents=True, exist_ok=True)

EXECUTABLE = PROJECT / "build" / ("criticality_sim.exe" if sys.platform.startswith("win") else "criticality_sim")
EXECUTABLE

## Run Controls

Set the flags to `True` to run the C/C++ simulator from this notebook. For long paper-scale runs, it is often better to run the VSCode tasks or PowerShell commands first, then keep these flags as `False` and only analyze existing outputs.

In [ ]:
RUN_EXACT_FIGURE_CONFIGS = False
RUN_PARAMETER_SWEEP = False

hysteresis_experiments = {
    "I0=0.3": PROJECT / "configs" / "experiments" / "hysteresis_i03.seed",
    "I0=0.8": PROJECT / "configs" / "experiments" / "hysteresis_i08.seed",
    "I0=1.1": PROJECT / "configs" / "experiments" / "hysteresis_i11.seed",
}
stationary_bin1_experiments = {
    "E0=5.7, I0=1.1": PROJECT / "configs" / "experiments" / "stationary_e57_i11_bin1.seed",
    "E0=6.8, I0=1.1": PROJECT / "configs" / "experiments" / "stationary_e68_i11_bin1.seed",
    "E0=6.9, I0=1.1": PROJECT / "configs" / "experiments" / "stationary_e69_i11_bin1.seed",
}
stationary_bin5_experiments = {
    "E0=5.7, I0=1.1": PROJECT / "configs" / "experiments" / "stationary_e57_i11_bin5.seed",
    "E0=6.8, I0=1.1": PROJECT / "configs" / "experiments" / "stationary_e68_i11_bin5.seed",
    "E0=6.9, I0=1.1": PROJECT / "configs" / "experiments" / "stationary_e69_i11_bin5.seed",
}

if RUN_EXACT_FIGURE_CONFIGS:
    for configs in [hysteresis_experiments, stationary_bin1_experiments, stationary_bin5_experiments]:
        for label, seed in configs.items():
            print("Running", label, seed.stem)
            run_simulation(EXECUTABLE, seed, RUN_ROOT, run_name=seed.stem)
else:
    print("Using existing exact-figure outputs under results/runs.")

## Fig.2: Hysteresis and Critical Transition

These panels use `smin/smax` sweeps of E0 at fixed I0. The notebook plots firing rate, overlap/order proxy, and Fano-like fluctuations against E0.

In [ ]:
for label, seed in hysteresis_experiments.items():
    run_name = seed.stem
    run_dir = RUN_ROOT / run_name
    medie = load_medie(find_output_file(run_dir, "medie3", run_name))
    q = load_q(find_output_file(run_dir, "q3", run_name), pout=20)
    combined = pd.merge_asof(medie.sort_values("time_ms"), q[["t_end", "q_max"]].rename(columns={"t_end": "time_ms"}).sort_values("time_ms"), on="time_ms")
    plot_hysteresis(combined, output=FIG_DIR / f"fig2_hysteresis_{run_name}.png")
    plt.suptitle(label, y=1.02)
    plt.show()

## Fig.3-Fig.4 Top Row: Parameter-Space Maps

The sweep grid below reproduces the paper's parameter-map logic. Increase the grid density for publication-grade maps; the defaults are intentionally editable because full dense sweeps are expensive.

In [ ]:
E0_VALUES = [5.7, 6.3, 6.8, 6.9, 7.2]
I0_VALUES = [0.3, 0.8, 1.1, 1.4]

sweep_configs = make_parameter_sweep(
    PROJECT / "configs" / "experiments" / "baseline_paper.seed",
    GENERATED_CONFIGS,
    E0_VALUES,
    I0_VALUES,
    prefix="paper_sweep",
    tmax=20000,
    bin=1,
    flush=30,
    pout=20,
    maxsp=4194304,
)

if RUN_PARAMETER_SWEEP:
    for name, seed in sweep_configs.items():
        print("Running sweep", name)
        run_simulation(EXECUTABLE, seed, RUN_ROOT, run_name=name)
else:
    print("Using existing sweep outputs if present.")

In [ ]:
summaries = []
missing = []
for name in sweep_configs:
    try:
        summaries.append(summarize_run(RUN_ROOT / name, name=name, pout=20))
    except FileNotFoundError:
        missing.append(name)

if missing:
    print(f"Missing {len(missing)} sweep runs. Set RUN_PARAMETER_SWEEP=True or run generated configs manually.")

summary = pd.DataFrame(summaries)
if not summary.empty:
    summary.to_csv(TABLE_DIR / "synthetic_parameter_sweep_summary.csv", index=False)
summary.head()

In [ ]:
if not summary.empty:
    for metric, filename in [
        ("q_mean", "fig3a_order_parameter_map.png"),
        ("q_var_mean", "fig3b_order_fluctuation_map.png"),
        ("fano_mean", "fig4a_fano_map.png"),
        ("module_isi_cv", "fig4b_module_isi_cv_map.png"),
        ("flexibility_n0", "fig4c_flexibility_map.png"),
    ]:
        plot_metric_heatmap(summary, metric, output=FIG_DIR / filename)
        plt.show()

## Fig.4 Examples: Raster, Module Rate, and Overlap

The three paper examples are subcritical E0=5.7 and critical E0=6.8/6.9 at I0=1.1.

In [ ]:
for label, seed in stationary_bin1_experiments.items():
    run_name = seed.stem
    run_dir = RUN_ROOT / run_name
    spikes = load_spikes(find_output_file(run_dir, "spikes3", run_name), pout=20)
    rate = load_rate(find_output_file(run_dir, "rate3", run_name))
    q = load_q(find_output_file(run_dir, "q3", run_name), pout=20)
    print(label)
    plot_raster(spikes, output=FIG_DIR / f"fig4_raster_{run_name}.png")
    plt.show()
    plot_module_rate(rate, output=FIG_DIR / f"fig4_module_rate_{run_name}.png")
    plt.show()
    plot_overlap(q, output=FIG_DIR / f"fig4_overlap_{run_name}.png")
    plt.show()

## Fig.5: Synthetic Neuronal Avalanches

Avalanches are continuous intervals where at least one module has nonzero rate in 5 ms bins. Size is the integral of summed module rates. The E0=6.9, I0=1.1 condition is fitted with the `powerlaw` package, matching the paper's method.

In [ ]:
avalanches = {}
for label, seed in stationary_bin5_experiments.items():
    run_name = seed.stem
    rate = load_rate(find_output_file(RUN_ROOT / run_name, "rate3", run_name))
    avalanches[label] = avalanche_table(rate)
    print(label, len(avalanches[label]), "avalanches")

fit_label = "E0=6.9, I0=1.1"
size_fit = fit_power_law(avalanches[fit_label]["size"])
duration_fit = fit_power_law(avalanches[fit_label]["duration_ms"])
fit_table = pd.DataFrame([
    {"quantity": "size", **size_fit.__dict__},
    {"quantity": "duration_ms", **duration_fit.__dict__},
])
fit_table.to_csv(TABLE_DIR / "fig5_powerlaw_fits.csv", index=False)
fit_table

In [ ]:
plot_avalanche_distribution(avalanches, "size", fit_label=fit_label, fit=size_fit, output=FIG_DIR / "fig5a_avalanche_size.png")
plt.show()
plot_avalanche_distribution(avalanches, "duration_ms", fit_label=fit_label, fit=duration_fit, output=FIG_DIR / "fig5b_avalanche_duration.png")
plt.show()

## Empirical Figures

Fig.6 and downstream empirical MEG/structure-function panels require source-reconstructed MEG and subject-level structural data. The current repository contains the embedded structural matrix used by the simulator, but not the empirical MEG time series, so those figures are not generated here.